# Week 8: isolated FedLoRA confirmation suites
Qwen2.5-1.5B, Kaggle T4 x2, one independent worker per GPU.
Confirmation: SST-2, QNLI, MNLI-m, MNLI-mm; seeds 6101-6106.
`METHOD_SUITE='factor_only'`: corrected FedAvg-LoRA and FFA-LoRA only, **48 confirmation jobs** or **2 smoke jobs**.
`METHOD_SUITE='spectral_only'`: FlexLoRA, FLoRIST and FLoRG only, **72 confirmation jobs** or **3 smoke jobs**.

These are paper-equation implementations adapted to the immediate-async Week 8 protocol, not reproductions of synchronous paper benchmarks. See `docs/week8/spectral_fedlora_fidelity_vi.md` and `docs/week8/fedavg_ffa_fidelity_vi.md`.
FLoRIST uses squared-energy tau=0.9. Shared rank caps and FLoRG rank-overflow limitations are documented.
Set `RUN_TRAINING=True` to launch. All cell outputs below are generated from real artifacts.
Recovery for notebook7b4f16575b: attach that notebook output through Add Input, keep RUN_TRAINING=False, and run all cells to rebuild reports. Completed jobs are removed from the queue. REQUIRE_RESUME=True prevents accidental full retraining when the previous output is not attached.


In [ ]:
from pathlib import Path
import json
import csv
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = 'https://github.com/TrgPhan/VASTLoRA.git'
REPO_REF = 'db4ca69024fdd0697f7ed667efbf10f025c558a3'  # Tested immutable code release.
RUN_MODE = 'confirmation'  # smoke / confirmation
METHOD_SUITE = 'factor_only'  # factor_only / spectral_only
RUN_TRAINING = False
SHARD_COUNT = 1
SHARD_INDEX = 0
GPU_IDS = [0, 1]
RESUME_ROOTS = ['/kaggle/input']  # Attach notebook7b4f16575b output via Add Input.
REQUIRE_RESUME = True  # Set False only for an intentional fresh experiment.
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'VASTLoRA-week8-baselines-v2'
if RUN_MODE not in {'smoke', 'confirmation'}:
    raise ValueError('Only smoke and confirmation are supported.')
if METHOD_SUITE not in {'factor_only', 'spectral_only'}:
    raise ValueError('Unknown method suite')
if SHARD_COUNT < 1 or not 0 <= SHARD_INDEX < SHARD_COUNT:
    raise ValueError('Invalid shard selection')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
if dirty:
    raise RuntimeError('Existing checkout contains changes; choose a fresh REPO_DIR.')
subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
required = ['scripts/week8_spectral_suite.py', 'tests/test_spectral_fedlora.py',
            'tests/test_week8_spectral_integration.py', 'configs/week8_spectral_requirements.txt',
            'src/riftlora/baselines/factor_averaging.py', 'tests/test_persistent_factor_baselines.py']
if any(not (REPO_DIR / p).exists() for p in required):
    raise RuntimeError('This GitHub revision does not contain the spectral suite. Publish the tested code first.')
# Kaggle can preinstall an old optional torchao that blocks PEFT import.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(REPO_DIR / 'configs/week8_spectral_requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[scale,dev]'], cwd=REPO_DIR, check=True)
RUNNER = REPO_DIR / 'scripts/run_week8_classification_matrix.py'
# pip install -e in a subprocess does not refresh this running kernel's sys.path.
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'scripts'))
import riftlora
if not Path(riftlora.__file__).resolve().is_relative_to((REPO_DIR / 'src').resolve()):
    raise RuntimeError('Kernel cached another riftlora checkout; restart the kernel.')
from week8_spectral_suite import build_matrix, job_list, SUITES
METHODS, SUITE_VERSION = SUITES[METHOD_SUITE]
expected_methods = {'factor_only': ('fedavg_lora', 'ffa_lora'), 'spectral_only': ('flexlora', 'florist', 'florg')}
if METHODS != expected_methods[METHOD_SUITE]:
    raise RuntimeError('Unexpected suite version or methods')
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_spectral_fedlora.py', 'tests/test_fed_lora_baselines.py',
                'tests/test_week8_spectral_integration.py', 'tests/test_persistent_factor_baselines.py'], cwd=REPO_DIR, check=True)
print('Release:', resolved_commit)

In [ ]:
matrix = build_matrix(smoke=RUN_MODE == 'smoke', suite=METHOD_SUITE)
MATRIX = WORK_ROOT / f'{SUITE_VERSION}_{RUN_MODE}_matrix.json'
MATRIX.write_text(json.dumps(matrix, indent=2), encoding='utf-8')
OUTPUT_ROOT = WORK_ROOT / f'week8_qwen15b_{SUITE_VERSION}_{RUN_MODE}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
all_jobs = job_list(matrix)
assert len(all_jobs) == len(METHODS) * (1 if RUN_MODE == 'smoke' else 24)
assert {job[2] for job in all_jobs} == set(METHODS)
jobs = all_jobs[SHARD_INDEX::SHARD_COUNT]
release_path = OUTPUT_ROOT / 'release.json'
release = {'commit': resolved_commit, 'suite': SUITE_VERSION, 'mode': RUN_MODE}
if release_path.exists() and json.loads(release_path.read_text()) != release:
    raise RuntimeError('Output directory belongs to another release; resume its commit or choose a new output root.')
release_path.write_text(json.dumps(release, indent=2), encoding='utf-8')
(OUTPUT_ROOT / 'matrix.json').write_text(json.dumps(matrix, indent=2), encoding='utf-8')
(OUTPUT_ROOT / f'job_plan_shard{SHARD_INDEX}.json').write_text(json.dumps(jobs, indent=2), encoding='utf-8')
subprocess.run([sys.executable, str(RUNNER), '--matrix', str(MATRIX),
                '--output-root', str(OUTPUT_ROOT), '--dry-run'], cwd=REPO_DIR, check=True)
print({'methods': matrix['methods'], 'tasks': [t['name'] for t in matrix['tasks']],
       'seeds': matrix['seeds'], 'total_jobs': len(all_jobs), 'shard_jobs': len(jobs)})

In [ ]:
# Only import directories belonging to the selected method suite.
from run_week8_classification_matrix import _build_config, _matrix_fingerprint, _completed_result_matches
tasks_by_name = {t['name']: t for t in matrix['tasks']}
regimes_by_name = {r['name']: r for r in matrix['regimes']}
def result_path_for(job):
    task, regime, method, seed = job
    return OUTPUT_ROOT / task / regime / method / f'{method}_seed{seed}' / 'result.json'
def completed_job(job, path=None):
    task, regime, method, seed = job
    path = result_path_for(job) if path is None else Path(path)
    spec = tasks_by_name[task]
    base = json.loads((REPO_DIR / spec['base_config']).read_text())
    cfg = _build_config(base, spec, regimes_by_name[regime], matrix)
    cfg['provenance'] = {'matrix_name': matrix['name'], 'matrix_sha256': _matrix_fingerprint(matrix),
                         'task': task, 'regime': regime, 'base_config': spec['base_config']}
    cfg['output_dir'] = str(result_path_for(job).parent.parent)
    if not path.exists() or not _completed_result_matches(path, config=cfg, method=method, seed=seed, matrix=matrix):
        return False
    counts = {'events.csv': cfg['experiment']['warmup_returns'] + cfg['experiment']['collected_returns'],
              'baseline_eval_details.csv': cfg['dataset']['eval_examples'],
              'final_eval_details.csv': cfg['dataset']['eval_examples']}
    try:
        for filename, count in counts.items():
            with (path.parent / filename).open(encoding='utf-8', newline='') as handle:
                if sum(1 for _ in csv.DictReader(handle)) != count:
                    return False
    except (OSError, csv.Error):
        return False
    return True
for source_root in RESUME_ROOTS:
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(source_root)
    for result in source_root.rglob('result.json'):
        try:
            payload = json.loads(result.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        if payload.get('method') not in METHODS:
            continue
        task, method, seed = payload.get('task'), payload['method'], payload.get('seed')
        regime = payload.get('regime')
        job = (task, regime, method, seed)
        if job not in all_jobs or not completed_job(job, result):
            continue
        if not completed_job(job):
            shutil.copytree(result.parent, result_path_for(job).parent, dirs_exist_ok=True)
completed_jobs = [job for job in all_jobs if completed_job(job)]
if REQUIRE_RESUME and not completed_jobs:
    raise RuntimeError('No valid previous results found. Attach notebook7b4f16575b output; training is blocked.')
jobs = [job for job in jobs if job not in completed_jobs]
(OUTPUT_ROOT / f'pending_jobs_shard{SHARD_INDEX}.json').write_text(json.dumps(jobs, indent=2))
print({'verified_completed': len(completed_jobs), 'pending_in_shard': len(jobs)})

In [ ]:
if RUN_TRAINING and jobs:
    # Cache once before concurrent workers; revision pins match the base config.
    from huggingface_hub import snapshot_download
    from datasets import load_dataset
    base = json.loads((REPO_DIR / 'configs/local_1_5b_rift_development.json').read_text())
    snapshot_download(base['model']['name'], revision=base['model']['revision'])
    for subset in sorted({t['subset'] for t in matrix['tasks']}):
        load_dataset(base['dataset']['hub_path'], subset, revision=base['dataset'].get('revision'))
    print('Model and dataset cache ready.')

In [ ]:
if RUN_TRAINING and jobs:
    import torch
    if len(GPU_IDS) != 2 or len(set(GPU_IDS)) != 2 or any(g < 0 or g >= torch.cuda.device_count() for g in GPU_IDS):
        raise RuntimeError('Select two available CUDA GPUs; Kaggle T4 x2 is expected.')
    log_dir = OUTPUT_ROOT / 'kaggle_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    failures = []
    for offset in range(0, len(jobs), 2):
        active = []
        try:
            for gpu, job in zip(GPU_IDS, jobs[offset:offset + 2]):
                task, regime, method, seed = job
                if method not in METHODS:
                    raise RuntimeError('Unexpected method in job queue')
                if completed_job(job):
                    print('skip verified', job, flush=True)
                    continue
                command = [sys.executable, '-u', str(RUNNER), '--matrix', str(MATRIX),
                           '--output-root', str(OUTPUT_ROOT), '--task', task,
                           '--regime', regime, '--method', method, '--seed', str(seed)]
                # Only an explicitly incomplete result may be overwritten; completed jobs were skipped above.
                if result_path_for(job).exists():
                    command.append('--force')
                log_path = log_dir / f'{task}_{method}_seed{seed}_gpu{gpu}.log'
                handle = log_path.open('a', encoding='utf-8')
                env = os.environ.copy()
                env.update(CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM='false',
                           PYTHONUNBUFFERED='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1',
                           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
                try:
                    process = subprocess.Popen(command, cwd=REPO_DIR, env=env,
                                               stdout=handle, stderr=subprocess.STDOUT)
                except BaseException:
                    handle.close()
                    raise
                active.append((process, handle, job, log_path))
                print('started', job, 'GPU', gpu, flush=True)
            while any(p.poll() is None for p, _, _, _ in active):
                time.sleep(5)
            for process, _, job, log_path in active:
                print({'job': job, 'return_code': process.returncode, 'log': str(log_path)}, flush=True)
                if process.returncode != 0:
                    failures.append({'job': job, 'return_code': process.returncode, 'log': str(log_path)})
        finally:
            for process, handle, _, _ in active:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=15)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                handle.close()
        (OUTPUT_ROOT / f'failures_shard{SHARD_INDEX}.json').write_text(json.dumps(failures, indent=2))
        if failures:
            raise RuntimeError('A job failed. Queue stopped; inspect failure JSON and logs before resuming.')
    print('Finished shard:', len(jobs))
else:
    print('No training launched. Pending jobs:', len(jobs), '; RUN_TRAINING:', RUN_TRAINING)

In [ ]:
# Report only results accepted by the same completion checker used for resume.
import pandas as pd
from run_week8_classification_matrix import _build_config, _matrix_fingerprint, _completed_result_matches
tasks = {t['name']: t for t in matrix['tasks']}
regimes = {r['name']: r for r in matrix['regimes']}
rows, pending = [], []
for task, regime, method, seed in all_jobs:
    spec = tasks[task]
    base = json.loads((REPO_DIR / spec['base_config']).read_text())
    cfg = _build_config(base, spec, regimes[regime], matrix)
    cfg['provenance'] = {'matrix_name': matrix['name'], 'matrix_sha256': _matrix_fingerprint(matrix),
                         'task': task, 'regime': regime, 'base_config': spec['base_config']}
    directory = OUTPUT_ROOT / task / regime / method
    cfg['output_dir'] = str(directory)
    path = directory / f'{method}_seed{seed}' / 'result.json'
    if not completed_job((task, regime, method, seed)):
        pending.append((task, method, seed))
        continue
    result = json.loads(path.read_text())
    m = result['metrics']
    rows.append({'task': task, 'method': method, 'seed': seed,
                 'Accuracy (%)': 100*m['final_accuracy'], 'Class NLL': m['final_class_nll'],
                 'Harmful (%)': 100*m['harmful_update_rate'],
                 'Late harmful (%)': 100*m['late_harmful_update_rate'],
                 'Runtime (min)': m['runtime_seconds']/60})
print('Valid results:', len(rows), '/', len(all_jobs), '; pending:', len(pending))
if rows:
    frame = pd.DataFrame(rows)
    frame.to_csv(OUTPUT_ROOT / 'per_seed_results.csv', index=False)
    summary = frame.groupby(['task', 'method']).agg(
        seeds=('seed', 'nunique'), accuracy=('Accuracy (%)', 'mean'),
        class_nll=('Class NLL', 'mean'), harmful=('Harmful (%)', 'mean'),
        late_harmful=('Late harmful (%)', 'mean'))
    display(summary)
    summary.to_csv(OUTPUT_ROOT / 'summary.csv')
print('Incomplete groups are preliminary; smoke results are diagnostic only.')
archive = WORK_ROOT / f'{SUITE_VERSION}_{RUN_MODE}_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file() and path.name != '.launcher.lock':
            z.write(path, path.relative_to(OUTPUT_ROOT.parent))
print('Archive:', archive)